# Lightweight HBCC — Tiny-ImageNet Training Pipeline

Trains all models on Tiny-ImageNet-200 (200 classes, 64×64 images resized to 32×32).

**Required Kaggle dataset:** `akash2sharma/tiny-imagenet`  
Add it via *Add Data → Datasets* before running. On Kaggle's filesystem it mounts at:
```
/kaggle/input/datasets/akash2sharma/tiny-imagenet/
  tiny-imagenet-200/
    train/<wnid>/images/<img>.JPEG    # 100 000 images, 500/class
    val/images/<img>.JPEG             # 10 000 images (labeled)
    val/val_annotations.txt
    wnids.txt
    words.txt
```

**Splits used:**
- Train: 90 000 images (90 % of official 100k train), carved with seed=42
- Val:   10 000 images (10 % of official train), carved with seed=42
- Test:  official val set (10 000 images, labeled via val_annotations.txt)

The official test split has no labels and is not used.

**Three sessions required** (~100 epochs each):

| Session | What runs | Est. T4 time |
|---------|-----------|-------------|
| **A** | ResNet-18 + MobileNetV2 + ShuffleNetV2 + CoC baselines | ~7 h |
| **B** | HBCC-Small CE + HBCC-Medium CE + Teacher re-train for KD | ~8 h |
| **C** | HBCC-Small KD + HBCC-Medium KD + Benchmarks | ~7 h |

Enable **only** the phases for the current session before committing.

In [ ]:
!git clone https://github.com/nvhungus/Lightweight-Context-Cluster.git
%cd Lightweight-Context-Cluster
!pip install fvcore -q

In [ ]:
# ── Locate Tiny-ImageNet dataset ───────────────────────────────────────────────
# Dataset: akash2sharma/tiny-imagenet (attached via Add Data → Datasets).
# Kaggle mounts it at /kaggle/input/datasets/akash2sharma/tiny-imagenet/
# The auto-detect below tries the known path first, then falls back to a walk.
from pathlib import Path as _Path
import os as _os

_tiny_root = None
_CANDIDATES = [
    # Primary: exact path for akash2sharma/tiny-imagenet
    _Path("/kaggle/input/datasets/akash2sharma/tiny-imagenet/tiny-imagenet-200"),
    # Fallbacks for other common Kaggle mount conventions
    _Path("/kaggle/input/tiny-imagenet/tiny-imagenet-200"),
    _Path("/kaggle/input/tiny-imagenet-200/tiny-imagenet-200"),
    _Path("/kaggle/input/tiny-imagenet"),
]
for _cand in _CANDIDATES:
    if (_cand / "train").is_dir():
        _tiny_root = _cand
        break

if _tiny_root is None:
    # Last resort: walk all of /kaggle/input/ looking for a train/ directory
    # that looks like Tiny-ImageNet (has wnids.txt alongside).
    for _dirpath, _dirs, _files in _os.walk("/kaggle/input"):
        _dp = _Path(_dirpath)
        if (_dp / "train").is_dir() and (_dp / "wnids.txt").exists():
            _tiny_root = _dp
            break

if _tiny_root is None:
    raise FileNotFoundError(
        "Could not locate tiny-imagenet-200/ under /kaggle/input/.\n"
        "Tried: " + ", ".join(str(c) for c in _CANDIDATES) + "\n"
        "Make sure 'akash2sharma/tiny-imagenet' is added as a dataset input."
    )

_TINY_ROOT = str(_tiny_root)
print(f"Tiny-ImageNet root : {_TINY_ROOT}")
print(f"Classes in train/  : {len(list(_tiny_root.joinpath('train').iterdir()))}")
print(f"val_annotations.txt: {(_tiny_root / 'val' / 'val_annotations.txt').exists()}")

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Repo root not found: {ROOT}"

PYTHON = Path(sys.executable)
print("Repo  :", ROOT)
print("Python:", PYTHON)

## Phase Switches

**Session A:** `RUN_ENV_CHECKS`, `RUN_TEACHER`, `RUN_LIGHTWEIGHT_BASELINES`, `RUN_COC_BASELINE` = True; all others False.

**Session B:** `RUN_STUDENT_CE`, `RUN_TEACHER_KD` = True; all others False.

**Session C:** `RUN_KD`, `RUN_BENCHMARKS` = True; all others False.

> **Cross-session metrics:** each session saves a CSV to `results/`.  
> In Session C, fill in `PREV_SESSION_A_METRICS` and `PREV_SESSION_B_METRICS`  
> so the final publication table includes all models.  
> Find CSV paths: committed session output → Output tab → hover over the CSV.

In [ ]:
# ── Session A ──────────────────────────────────────────────────────────────────
RUN_ENV_CHECKS            = True   # run only in Session A
RUN_TEACHER               = True   # ResNet-18 reference baseline
RUN_LIGHTWEIGHT_BASELINES = True   # MobileNetV2, ShuffleNetV2
RUN_COC_BASELINE          = True   # CoC pure-cluster baseline

# ── Session B ──────────────────────────────────────────────────────────────────
RUN_STUDENT_CE            = False  # HBCC-Small CE + HBCC-Medium CE  (~5.8 h)
RUN_TEACHER_KD            = False  # re-train ResNet-18 as KD teacher (~1.7 h)

# ── Session C ──────────────────────────────────────────────────────────────────
RUN_KD                    = False  # HBCC-Small KD + HBCC-Medium KD  (~6.1 h)
RUN_BENCHMARKS            = False  # latency / params / MACs          (~0.5 h)

# ── Epoch budgets ───────────────────────────────────────────────────────────────
# Tiny-ImageNet: 90k effective train images → ~703 batches/epoch at batch_size=128.
# That is ~1.3× Food-101 and ~2× CIFAR-10 per epoch.
# Conservative T4 estimates (scale Food-101 actuals × 1.32):
#   Session A: env(~0.3h) + ResNet18(~1.7h) + MobileNetV2(~1.7h)
#              + ShuffleNetV2(~1.3h) + CoC(~2.0h) ≈ 7.0 h
#   Session B: HBCC-Small CE(~2.9h) + HBCC-Medium CE(~3.2h)
#              + Teacher KD re-train(~1.7h)       ≈ 7.8 h
#   Session C: HBCC-Small KD(~2.9h) + HBCC-Medium KD(~3.2h)
#              + benchmarks(~0.5h)                ≈ 6.6 h
BASELINE_EPOCHS = 100
HBCC_CE_EPOCHS  = 100
KD_EPOCHS       = 100

# ── Reproducible seed ──────────────────────────────────────────────────────────
SEED = 42
COMMON_TRAIN_OVERRIDES = [f"train.seed={SEED}"]

# ── Dataset ────────────────────────────────────────────────────────────────────
# _TINY_ROOT is set in cell t003 (auto-detected from /kaggle/input/).
# data.py accepts root pointing directly to tiny-imagenet-200/.
# data.val_size=10000 overrides the YAML default (5000, from CIFAR configs)
# so we get 90k train / 10k val (50 images per class for validation).
TINY_BASE = [
    "data.name=tiny_imagenet",
    f"data.root={_TINY_ROOT}",
    "model.num_classes=200",
    "data.val_size=10000",
    "data.workers=4",
]

# ── Benchmark settings ─────────────────────────────────────────────────────────
BENCHMARK_OUTPUT        = "results/benchmark_tiny_imagenet"
BENCHMARK_BATCHES       = [1, 16, 64, 128]
BENCHMARK_WARMUP        = 30
BENCHMARK_RUNS          = 100
DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP  = 2
DEBUG_BENCHMARK_RUNS    = 3

# ── Cross-session metrics (Session C only) ─────────────────────────────────────
# After committing Sessions A and B, paste CSV paths here before running Session C.
PREV_SESSION_A_METRICS = ""  # "/kaggle/input/<dataset-a>/results/tiny_imagenet_session_a.csv"
PREV_SESSION_B_METRICS = ""  # "/kaggle/input/<dataset-b>/results/tiny_imagenet_session_b.csv"

In [ ]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def tiny_overrides(
    experiment_name: str | None = None,
    extra_overrides: list[str] | None = None,
) -> list[str]:
    items = [*TINY_BASE]
    if experiment_name is not None:
        items.append(f"experiment.name={experiment_name}")
    items.extend(extra_overrides or [])
    return items


def train(
    config: str,
    output: str,
    experiment_name: str,
    overrides: list[str] | None = None,
    extra: list[str] | None = None,
) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args(tiny_overrides(experiment_name, [*(overrides or []), *COMMON_TRAIN_OVERRIDES]))
    args += extra or []
    run_py(args)


def benchmark(
    config: str,
    experiment_name: str,
    checkpoint: str | None = None,
    debug: bool = False,
    profile: bool = True,
) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup  = DEBUG_BENCHMARK_WARMUP  if debug else BENCHMARK_WARMUP
    runs    = DEBUG_BENCHMARK_RUNS    if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py", "--config", config,
        "--output", BENCHMARK_OUTPUT,
        "--batch-sizes", *[str(v) for v in batches],
        "--warmup", str(warmup),
        "--runs", str(runs),
    ]
    args += override_args(["model.num_classes=200", f"experiment.name={experiment_name}"])
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"[warn] Checkpoint not found — benchmarking architecture only: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def save_session_metrics(session_label: str, run_dirs: list[str]) -> Path:
    rows = []
    for run_root in run_dirs:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record  = test_records[-1] if test_records else {}
            rows.append({
                "experiment":    metrics_path.parent.name,
                "best_val_acc1": best.get("val_acc1"),
                "best_val_acc5": best.get("val_acc5"),
                "test_acc1":     test_record.get("test_acc1"),
                "test_acc5":     test_record.get("test_acc5"),
            })
    out_dir = ROOT / "results"
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / f"{session_label}.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f"Saved {len(rows)} rows → {out_path}")
    return out_path

## 0. Environment Checks

Run only in **Session A** (`RUN_ENV_CHECKS = True`).  
Verifies the Tiny-ImageNet directory structure, class count, and split sizes,  
then runs the unit-test suite and traces HBCC-Medium shapes for 200 classes.

In [ ]:
if RUN_ENV_CHECKS:
    from pathlib import Path as _P
    _tr = _P(_TINY_ROOT)
    _train_dir = _tr / "train"
    _val_ann   = _tr / "val" / "val_annotations.txt"
    _wnids     = _tr / "wnids.txt"

    assert _train_dir.is_dir(), f"train/ not found: {_train_dir}"
    assert _val_ann.exists(),   f"val/val_annotations.txt not found: {_val_ann}"
    assert _wnids.exists(),     f"wnids.txt not found: {_wnids}"

    _classes    = sorted(d.name for d in _train_dir.iterdir() if d.is_dir())
    _train_imgs = sum(1 for _ in _train_dir.glob("*/images/*.JPEG"))
    _val_lines  = [l for l in _val_ann.read_text().splitlines() if l.strip()]

    print(f"Classes     : {len(_classes)}")
    print(f"Train images: {_train_imgs}")
    print(f"Val entries : {len(_val_lines)}")

    assert len(_classes) == 200,      f"Expected 200 classes, found {len(_classes)}"
    assert _train_imgs == 100_000,    f"Expected 100 000 train images, got {_train_imgs}"
    assert len(_val_lines) == 10_000, f"Expected 10 000 val entries, got {len(_val_lines)}"
    print("OK: Tiny-ImageNet verified (200 classes, 100k train, 10k val)")

    run_py(["-m", "pytest", "-q"])

    run_py([
        "tools/shape_trace.py", "--config", "configs/hbcc_accuracy_medium.yaml",
        "--batch-size", "1",
        *override_args(["model.num_classes=200"]),
    ])

## Session A — Baselines (~7 h)

ResNet-18 (teacher), MobileNetV2, ShuffleNetV2, and CoC, all for `BASELINE_EPOCHS` epochs.

After committing: download `results/tiny_imagenet_session_a.csv` from the Output tab.

In [ ]:
EXP_RESNET  = "resnet18_tin"
DIR_TEACHER = "runs_tin_teacher"

if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER,
        EXP_RESNET,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )

In [ ]:
EXP_MOBILENET  = "mobilenet_v2_tin"
EXP_SHUFFLENET = "shufflenet_v2_x1_0_tin"
DIR_BASELINES  = "runs_tin_baselines"

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        DIR_BASELINES,
        EXP_MOBILENET,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        DIR_BASELINES,
        EXP_SHUFFLENET,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )

In [ ]:
EXP_COC = "coc_baseline_tin"
DIR_COC = "runs_tin_coc"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        DIR_COC,
        EXP_COC,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )

In [ ]:
if any([RUN_TEACHER, RUN_LIGHTWEIGHT_BASELINES, RUN_COC_BASELINE]):
    save_session_metrics("tiny_imagenet_session_a", [DIR_TEACHER, DIR_BASELINES, DIR_COC])

## Session B — HBCC CE + Teacher re-train (~8 h)

1. **HBCC-Small CE** (~2.9 h)
2. **HBCC-Medium CE** (~3.2 h)
3. **ResNet-18 KD teacher re-train** (~1.7 h)

> **To skip the teacher re-train (~1.7 h):** set `RUN_TEACHER_KD = False` and point
> `TEACHER_KD_CHECKPOINT` to the Session A ResNet-18 checkpoint added as a dataset input:
> ```python
> TEACHER_KD_CHECKPOINT = "/kaggle/input/<dataset>/runs_tin_teacher/resnet18_tin/best.pth"
> ```

After committing: download `results/tiny_imagenet_session_b.csv`.

In [ ]:
EXP_HBCC_SML = "hbcc_accuracy_small_tin"
EXP_HBCC_MED = "hbcc_accuracy_medium_tin"
DIR_ACCURACY = "runs_tin_ce"

if RUN_STUDENT_CE:
    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_ACCURACY,
        EXP_HBCC_SML,
        overrides=[f"train.epochs={HBCC_CE_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_ACCURACY,
        EXP_HBCC_MED,
        overrides=[f"train.epochs={HBCC_CE_EPOCHS}"],
    )

In [ ]:
EXP_RESNET_KD  = "resnet18_tin_kd"
DIR_TEACHER_KD = "runs_tin_teacher_kd"

# Default: checkpoint produced by RUN_TEACHER_KD in this session (Session B).
TEACHER_KD_CHECKPOINT = f"{DIR_TEACHER_KD}/{EXP_RESNET_KD}/best.pth"

# ── Session C: override this with an absolute path from a Kaggle dataset input ──
# After committing Session B, add its output as a dataset input in Session C,
# then uncomment one of the lines below:
#
# Option 1 — use Session B's teacher (recommended, trained on Tiny-ImageNet):
# TEACHER_KD_CHECKPOINT = "/kaggle/input/<session-b-output>/runs_tin_teacher_kd/resnet18_tin_kd/best.pth"
#
# Option 2 — reuse Session A's teacher (saves ~1.7 h of re-training in Session B):
# TEACHER_KD_CHECKPOINT = "/kaggle/input/<session-a-output>/runs_tin_teacher/resnet18_tin/best.pth"
#
# Note: pathlib handles absolute paths correctly, so ROOT / absolute_path = absolute_path.

if RUN_TEACHER_KD:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER_KD,
        EXP_RESNET_KD,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )

In [ ]:
if any([RUN_STUDENT_CE, RUN_TEACHER_KD]):
    save_session_metrics("tiny_imagenet_session_b", [DIR_ACCURACY, DIR_TEACHER_KD])

## Session C — HBCC KD + Benchmarks (~7 h)

1. **HBCC-Small KD** (~2.9 h)
2. **HBCC-Medium KD** (~3.2 h)
3. **Benchmarks** (~0.5 h)

> **Before running:** fill in `PREV_SESSION_A_METRICS` and `PREV_SESSION_B_METRICS`  
> in the phase-switches cell so the final publication table includes all models.

In [ ]:
EXP_HBCC_SML_KD = "hbcc_accuracy_small_kd_tin"
EXP_HBCC_MED_KD = "hbcc_accuracy_medium_kd_tin"
DIR_KD          = "runs_tin_kd"

KD_OVERRIDES = ["train.kd_alpha=0.5", "train.kd_temperature=4.0"]

if RUN_KD:
    teacher_path = ROOT / TEACHER_KD_CHECKPOINT
    if not teacher_path.exists():
        raise FileNotFoundError(
            f"Teacher checkpoint not found: {teacher_path}\n"
            "Set RUN_TEACHER_KD=True in Session B, or point TEACHER_KD_CHECKPOINT "
            "to an existing checkpoint."
        )
    print(f"Teacher checkpoint: {teacher_path}")

    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_KD,
        EXP_HBCC_SML_KD,
        overrides=[f"train.epochs={KD_EPOCHS}", *KD_OVERRIDES],
        extra=[
            "--teacher-config",     "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint", TEACHER_KD_CHECKPOINT,
        ],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_KD,
        EXP_HBCC_MED_KD,
        overrides=[f"train.epochs={KD_EPOCHS}", *KD_OVERRIDES],
        extra=[
            "--teacher-config",     "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint", TEACHER_KD_CHECKPOINT,
        ],
    )

In [ ]:
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml",            EXP_RESNET,       None),
    ("configs/baselines/mobilenet_v2_cifar.yaml",        EXP_MOBILENET,    None),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml", EXP_SHUFFLENET,   None),
    ("configs/coc_cifar_baseline.yaml",                  EXP_COC,          None),
    ("configs/hbcc_accuracy_small.yaml",                 EXP_HBCC_SML,     None),
    ("configs/hbcc_accuracy_medium.yaml",                EXP_HBCC_MED,     None),
    ("configs/hbcc_accuracy_small.yaml",                 EXP_HBCC_SML_KD,  f"{DIR_KD}/{EXP_HBCC_SML_KD}/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",                EXP_HBCC_MED_KD,  f"{DIR_KD}/{EXP_HBCC_MED_KD}/best.pth"),
]

if RUN_BENCHMARKS:
    for cfg, exp_name, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print(f"Skip missing config: {cfg}")
            continue
        benchmark(cfg, exp_name, checkpoint=ckpt)

In [ ]:
if any([RUN_KD, RUN_BENCHMARKS]):
    save_session_metrics("tiny_imagenet_session_c", [DIR_KD])

## Results Tables

Run in any session. For the complete publication table, run in Session C with
`PREV_SESSION_A_METRICS` and `PREV_SESSION_B_METRICS` filled in.

In [ ]:
TIN_RUN_DIRS = [
    DIR_TEACHER, DIR_BASELINES, DIR_COC,
    DIR_ACCURACY, DIR_TEACHER_KD, DIR_KD,
]


def collect_train_metrics() -> pd.DataFrame:
    rows = []
    for run_root in TIN_RUN_DIRS:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record  = test_records[-1] if test_records else {}
            rows.append({
                "experiment":    metrics_path.parent.name,
                "best_val_acc1": best.get("val_acc1"),
                "best_val_acc5": best.get("val_acc5"),
                "test_acc1":     test_record.get("test_acc1"),
                "test_acc5":     test_record.get("test_acc5"),
            })

    _cols = ["experiment", "best_val_acc1", "best_val_acc5", "test_acc1", "test_acc5"]
    df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)

    for prev_path in [PREV_SESSION_A_METRICS, PREV_SESSION_B_METRICS]:
        if prev_path:
            p = Path(prev_path)
            if p.exists():
                prev_df  = pd.read_csv(p)
                existing = set(df["experiment"])
                new_rows = prev_df[~prev_df["experiment"].isin(existing)]
                df = pd.concat([df, new_rows], ignore_index=True)
                print(f"Merged {len(new_rows)} rows from {p.name}")
            else:
                print(f"[warn] Not found (skipped): {p}")

    if df.empty:
        return df
    return df.sort_values("best_val_acc1", ascending=False).reset_index(drop=True)


train_df = collect_train_metrics()
train_df

In [ ]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / BENCHMARK_OUTPUT).glob("*.json"):
        try:
            rows.append(json.loads(path.read_text(encoding="utf-8")))
        except json.JSONDecodeError:
            print(f"[warn] Could not parse: {path.name}")
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
arch_cols = ["config_id", "model_name", "params_total", "macs", "latency_ms_b1", "peak_memory_mb"]
bench_df[[c for c in arch_cols if c in bench_df.columns]] if not bench_df.empty else bench_df

In [ ]:
def build_publication_table(bench: pd.DataFrame, train: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        print("No benchmark data yet — run Session C with RUN_BENCHMARKS=True.")
        return bench
    out = bench.copy()
    acc_map, acc5_map = {}, {}
    if not train.empty:
        acc_map  = dict(zip(train["experiment"], train["test_acc1"].fillna(train["best_val_acc1"])))
        if "test_acc5" in train.columns:
            acc5_map = dict(zip(train["experiment"], train["test_acc5"].fillna(train["best_val_acc5"])))
    out["acc1"] = out["config_id"].map(acc_map)
    out["acc5"] = out["config_id"].map(acc5_map)
    rename = {
        "config_id":      "Model",
        "acc1":           "Top-1 Acc (%)",
        "acc5":           "Top-5 Acc (%)",
        "params_total":   "Params",
        "macs":           "MACs",
        "latency_ms_b1":  "Latency b1 (ms)",
        "peak_memory_mb": "Peak Memory (MB)",
    }
    cols  = [c for c in rename if c in out.columns]
    table = out[cols].rename(columns=rename).dropna(subset=["Top-1 Acc (%)"]).sort_values("Top-1 Acc (%)", ascending=False)
    return table.reset_index(drop=True)


pub_df = build_publication_table(bench_df, train_df)
out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
pub_df.to_csv(out_dir / "tiny_imagenet_published_results.csv", index=False)
pub_df

## Recommended Execution Order

**Session A (~7 h):**
1. Set `RUN_ENV_CHECKS`, `RUN_TEACHER`, `RUN_LIGHTWEIGHT_BASELINES`, `RUN_COC_BASELINE` = True; all others False.
2. Run all cells and commit.
3. Download `results/tiny_imagenet_session_a.csv` from the Output tab.

**Session B (~8 h, or ~6.3 h skipping teacher re-train):**
1. Set `RUN_STUDENT_CE` = True, `RUN_TEACHER_KD` = True; all others False.
2. *(Optional, saves ~1.7 h)* Add Session A's committed output as a Kaggle dataset input,
   set `RUN_TEACHER_KD = False`, and in cell t013 uncomment Option 2 with the correct path.
3. Run all cells and commit.
4. Download `results/tiny_imagenet_session_b.csv`.

**Session C (~6.6 h):**
1. Set `RUN_KD` = True, `RUN_BENCHMARKS` = True; all others False.
2. Paste Session A and B CSV paths into `PREV_SESSION_A_METRICS` / `PREV_SESSION_B_METRICS`.
3. Add Session B's committed output as a Kaggle dataset input.
   In cell t013, uncomment **Option 1** and replace `<session-b-output>` with the dataset name
   that Kaggle assigned (visible in the right panel → Data).
4. Run all cells and commit.

The publication table and `tiny_imagenet_published_results.csv` are written to `results/` at the end of Session C.